#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "requires_maintenance"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")
df.display()

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Null handling and Empty values

In [0]:
def is_empty(col):
    return col.isNull() | (F.trim(col) == "")

condition_invalid = (
    is_empty(F.col("ID")) |
    is_empty(F.col("CAT")) |
    is_empty(F.col("SUBCAT")) |
    is_empty(F.col("MAINTENANCE"))
)

df_valid = df.filter(~condition_invalid)
df_invalid = df.filter(condition_invalid)

invalid_count = df_invalid.count()

print("Total rows:", df.count())
print("Invalid rows:", df_invalid.count())
print("Valid rows:", df_valid.count())

df = df_valid

## Handle duplicate ids

In [0]:
duplicate_id_count = (
    df.groupBy("ID")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print("Duplicate ID:", duplicate_id_count)

if duplicate_id_count > 0:
    raise Exception("Duplicate category_id found!")


## Renamig the columns

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks before write

In [0]:
def sanity_check(df):
    row_count = df.count()

    duplicate_category_id = (
        df.groupBy("category_id")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    null_critical_fields = (
        df.filter(
            F.col("category_id").isNull() |
            F.col("category").isNull() |
            F.col("subcategory").isNull() |
            F.col("requires_maintenance").isNull()
        )
        .count()
    )

    invalid_maintenance = (
        df.filter(
            ~F.col("requires_maintenance").isin("Yes", "No", "Unknown")
        )
        .count()
    )

    return {
        "row_count": row_count,
        "duplicate_category_id": duplicate_category_id,
        "null_critical_fields": null_critical_fields,
        "invalid_maintenance": invalid_maintenance
    }

results = sanity_check(df)
print("Before write:", results)

# Write Into Silver

In [0]:
(df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.erp_product_categories"))

df_silver = spark.table("workspace.silver.erp_product_categories")

results = sanity_check(df_silver)
print("After write:", results)

if results["duplicate_category_id"] > 0:
    raise Exception("Duplicate category_id found!")

if results["null_critical_fields"] > 0:
    raise Exception("Null critical fields found!")

if results["invalid_maintenance"] > 0:
    raise Exception("Invalid_maintenance values found!")
